# ACIT4620 | Reproducible CNN–Sugeno Neuro-Fuzzy Skin-Lesion Classification

**Self-contained training and evaluation pipeline — HAM10000 only**

The input dataset is the original HAM10000 release (metadata CSV and its two official image directories). This notebook **creates its own lesion-grouped partition** from raw metadata and saves its manifest in the output directory. It never imports previously trained CNN or ANFIS models, features, predictions, historical manifests, or experimental scores.

**Architecture:** ImageNet-pretrained EfficientNet-B0 → 16-dimensional projection → compact first-order Sugeno ANFIS with 16 Gaussian rules. A CNN-only classifier is retained as a controlled baseline; no MLP experiment is included.

**Scientific protocol:** label-aware lesion-grouped split created once with a fixed seed; training-only checkpoint selection; refit on the training partition; validation summaries; end-to-end training-only grouped cross-validation; a single final test evaluation of the predeclared coverage-regularized hybrid only after all training-only CV folds finish. The test is not used to choose models or hyperparameters. The split is newly generated and is **not identical to previously used study partitions**.

**Input:** `HAM10000_metadata.csv`, `HAM10000_images_part_1/`, `HAM10000_images_part_2/` (available together in the Kaggle Skin Cancer MNIST: HAM10000 dataset). ImageNet weights are obtained from torchvision when internet/cache is available; an official pretrained weight file is optional for offline use. No experiment-specific data uploads are required.

Outputs include the generated split manifest, training curves, metrics, per-image predictions, model checkpoints, reproducibility metadata, fuzzy-rule diagnostics, grouped CV, explanation maps, and downloadable report/checkpoint archives.

**Predeclared primary model:** Coverage-regularized Sugeno ANFIS, selected because it obtained the highest mean training-only CV macro-F1 during protocol development. The CNN and ordinary ANFIS remain development baselines. The final test is evaluated **once**, with results saved separately from validation.


## 0. Setup and experimental configuration

Run from top to bottom in a fresh Kaggle session. Select a GPU accelerator for full training. The default experiment performs the complete three-fold training-only CV after the final models and backs up results after each fold. Full execution is computationally expensive. **Do not interpret any score from previous splits as a benchmark on this new partition.**


In [ ]:
from __future__ import annotations

import copy
import gc
import hashlib
import json
import os
import platform
import random
import tempfile
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, log_loss,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from IPython.display import display

@dataclass(frozen=True)
class Config:
    mode: str = "train"  # always train from ImageNet pretrained initialization
    run_cv: bool = True
    run_gradcam: bool = True
    run_quality_audit: bool = True
    run_roi_audit: bool = False
    approve_roi: bool = False
    image_size: int = 300
    batch_size: int = 8
    feature_batch_size: int = 16
    num_workers: int = 0
    seed: int = 42
    image_net_weights: str | None = None  # official ImageNet EfficientNet-B0 .pth, if offline
    dataset_root: str | None = None
    split_seed: int = 2026  # controls a NEW lesion-grouped split from raw metadata
    split_folds: int = 7  # 5 train folds, 1 validation fold, 1 sealed-test fold
    final_test_once: bool = True  # predeclared, one-time test evaluation after CV; never tune on test
    final_test_model: str = "Sugeno_ANFIS_coverage"  # chosen via training-only CV macro-F1
    output_dir: str = "/kaggle/working/acit4620_raw_ham10000_study" if Path("/kaggle/working").exists() else "./acit4620_raw_ham10000_study"
    max_epochs: int = 20
    patience: int = 6
    warmup_epochs: int = 2
    unfreeze_blocks: int = 4
    learning_rate_backbone: float = 3e-5
    learning_rate_classifier: float = 3e-4
    learning_rate_anfis: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = .02
    use_amp: bool = True
    data_policy: str = "standard"  # standard | sqrt_sampler | sqrt_weighted | strong_aug | wide_roi
    fuzzy_rules: int = 16
    coverage_lambda: float = .02
    head_max_epochs: int = 50
    head_patience: int = 8
    cv_folds: int = 3
    cv_cnn_epochs: int = 16
    cv_anfis_epochs: int = 15
    bootstrap_repeats: int = 300
    run_data_policy_ablation: bool = False  # extra train-only CNN comparisons; expensive
    ablation_epochs: int = 8
    cleanlab_review: bool = False  # only label review on training-only OOF predictions

CFG = Config()
CLASSES = ("nv", "mel", "bkl", "bcc", "akiec", "vasc", "df")
EXPECTED_IMAGES = 10015
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path(CFG.output_dir).expanduser().resolve()
for path in (OUT, OUT / "figures", OUT / "tables", OUT / "checkpoints", OUT / "cv"):
    path.mkdir(parents=True, exist_ok=True)
assert CFG.mode == "train", "This is a from-scratch training notebook"
assert CFG.image_size == 300 and CFG.unfreeze_blocks == 4 and CFG.warmup_epochs == 2
assert CFG.data_policy in {"standard", "sqrt_sampler", "sqrt_weighted", "strong_aug", "wide_roi"}
assert CFG.fuzzy_rules >= len(CLASSES) and CFG.bootstrap_repeats >= 0
assert CFG.cv_folds == 3 and CFG.cv_cnn_epochs > CFG.warmup_epochs
assert CFG.split_folds == 7 and CFG.final_test_model in {"CNN_300_deeper", "Sugeno_ANFIS", "Sugeno_ANFIS_coverage"}
print("Mode:", CFG.mode, "| Device:", DEVICE, "| Output:", OUT)
print("Group CV:", CFG.run_cv, "| CNN initialization:",
      "ImageNet transfer learning (no prior HAM10000 checkpoints)")


## 1. Reproducible split generation and lesion-leakage safeguards

This section reads only the original HAM10000 metadata; it generates a *new* lesion-disjoint split using stratified grouped folds. Label information is used solely for split design. The test partition is assigned here but its image files are **not indexed or decoded** during model development. The split manifest is saved as an output and checked against the original raw-metadata fingerprint on subsequent executions.


In [ ]:
def find_unique(name: str, roots=(Path("/kaggle/input"), Path("./data"))):
    """Find one canonical file rather than silently choosing a duplicate."""
    matches = sorted({p.resolve() for root in roots if root.exists()
                      for p in root.rglob(name) if p.is_file()})
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one {name}; found {len(matches)}: {matches[:6]}")
    return matches[0]


def sha256_file(path: Path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def choose_lesion_grouped_partition(metadata: pd.DataFrame, seed: int, folds: int = 7):
    """Create a deterministic ~5:1:1 train/validation/test split by lesion group.

    Candidate holdout folds are scored using *metadata only* for approximate
    sample-size and label-proportion balance. Neither image pixels nor model
    predictions influence the split.
    """
    data = metadata.reset_index(drop=True).copy()
    expected_classes = set(CLASSES)
    splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    heldout = [np.asarray(eval_idx, dtype=int) for _, eval_idx in
               splitter.split(data, y=data["dx"], groups=data["lesion_id"])]
    n = len(data)
    overall = data.dx.value_counts(normalize=True).reindex(CLASSES).to_numpy()
    best = None
    for valid_fold in range(folds):
        for test_fold in range(folds):
            if valid_fold == test_fold:
                continue
            va, te = heldout[valid_fold], heldout[test_fold]
            tr = np.setdiff1d(np.arange(n), np.concatenate([va, te]))
            if any(set(data.iloc[idx].dx) != expected_classes for idx in (tr, va, te)):
                continue
            size_cost = abs(len(va) / n - 1 / folds) + abs(len(te) / n - 1 / folds)
            distribution_cost = sum(np.abs(
                data.iloc[idx].dx.value_counts(normalize=True)
                .reindex(CLASSES, fill_value=0).to_numpy() - overall).sum()
                for idx in (va, te))
            score = size_cost * 5 + distribution_cost
            if best is None or score < best[0]:
                best = (score, tr, va, te)
    if best is None:
        raise ValueError("No leakage-safe split with all seven classes; inspect metadata")
    _, tr, va, te = best
    role = np.full(n, "", dtype="<U5")
    role[tr] = "train"
    role[va] = "val"
    role[te] = "test"
    result = data[["image_id", "lesion_id"]].copy()
    result["split"] = role
    assert (role != "").all() and result.image_id.is_unique
    group_sets = {name: set(result.loc[result.split == name, "lesion_id"])
                  for name in ("train", "val", "test")}
    assert group_sets["train"].isdisjoint(group_sets["val"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["val"].isdisjoint(group_sets["test"])
    assert result.groupby("lesion_id").split.nunique().eq(1).all()
    return result


META_PATH = find_unique("HAM10000_metadata.csv")
DATA_ROOT = Path(CFG.dataset_root) if CFG.dataset_root else META_PATH.parent
METADATA_SHA256 = sha256_file(META_PATH)
metadata_raw = pd.read_csv(META_PATH)
required_cols = {"image_id", "lesion_id", "dx"}
assert required_cols.issubset(metadata_raw.columns), required_cols - set(metadata_raw.columns)
assert len(metadata_raw) == EXPECTED_IMAGES, "HAM10000 should contain 10,015 metadata rows"
assert metadata_raw.image_id.is_unique and metadata_raw.lesion_id.notna().all()
assert set(metadata_raw.dx) == set(CLASSES)
metadata_raw = metadata_raw.reset_index(drop=True)

MANIFEST_PATH = OUT / "tables" / "generated_lesion_grouped_split.csv"
SPLIT_INFO_PATH = OUT / "tables" / "split_generation.json"
if MANIFEST_PATH.exists():
    if not SPLIT_INFO_PATH.exists():
        raise RuntimeError("Split provenance missing; use a fresh output_dir")
    previous = json.loads(SPLIT_INFO_PATH.read_text())
    assert previous["metadata_sha256"] == METADATA_SHA256
    assert previous["split_seed"] == CFG.split_seed
    assert previous["split_folds"] == CFG.split_folds
    assert previous["generated_manifest_sha256"] == sha256_file(MANIFEST_PATH), (
        "Saved split changed; use a new output directory")
    partition = pd.read_csv(MANIFEST_PATH)
else:
    partition = choose_lesion_grouped_partition(
        metadata_raw, CFG.split_seed, CFG.split_folds)
    temporary = MANIFEST_PATH.with_suffix(".tmp.csv")
    partition.to_csv(temporary, index=False)
    os.replace(temporary, MANIFEST_PATH)
    provenance = {"metadata_sha256": METADATA_SHA256,
                  "split_seed": CFG.split_seed, "split_folds": CFG.split_folds,
                  "method": "StratifiedGroupKFold; label-only selection of validation/test folds",
                  "dataset_images": len(metadata_raw),
                  "split_sizes": partition.split.value_counts().to_dict(),
                  "generated_manifest_sha256": sha256_file(MANIFEST_PATH)}
    temporary_info = SPLIT_INFO_PATH.with_suffix(".tmp.json")
    temporary_info.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    os.replace(temporary_info, SPLIT_INFO_PATH)

assert set(partition.split) == {"train", "val", "test"}
assert set(partition.image_id) == set(metadata_raw.image_id)
assert partition.image_id.is_unique
assert partition.groupby("lesion_id").split.nunique().eq(1).all()
MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
lesion_sets = {key: set(partition.loc[partition.split == key, "lesion_id"])
               for key in ("train", "val", "test")}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert lesion_sets[a].isdisjoint(lesion_sets[b])

# Only development rows carry image paths; test image pixels remain untouched.
train_val_ids = set(partition.loc[partition.split != "test", "image_id"])
meta_dev = metadata_raw[metadata_raw.image_id.isin(train_val_ids)].copy()
DEV = partition.loc[partition.split != "test"].merge(
    meta_dev.drop(columns="lesion_id"), on="image_id", validate="one_to_one")
DEV = DEV.reset_index(drop=True)
assert DEV.dx.notna().all() and set(DEV.dx) == set(CLASSES)

# The kmader mirror contains lowercase duplicate folders. Index official uppercase folders only.
image_dirs = [DATA_ROOT / "HAM10000_images_part_1", DATA_ROOT / "HAM10000_images_part_2"]
assert all(folder.is_dir() for folder in image_dirs), (
    f"Missing canonical uppercase HAM10000 image folders in {DATA_ROOT}")
image_index = {}
for folder in image_dirs:
    for path in folder.glob("*.jpg"):
        if path.stem in train_val_ids:
            if path.stem in image_index:
                raise RuntimeError(f"Duplicate canonical image_id: {path.stem}")
            image_index[path.stem] = str(path)
assert set(image_index) == train_val_ids, "Training/validation image indexing incomplete"
DEV["path"] = DEV.image_id.map(image_index)
TRAIN = DEV.loc[DEV.split == "train"].copy().reset_index(drop=True)
VALID = DEV.loc[DEV.split == "val"].copy().reset_index(drop=True)
TEST_IDS = partition.loc[partition.split == "test", "image_id"].copy()
assert len(TRAIN) + len(VALID) + len(TEST_IDS) == EXPECTED_IMAGES

inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
inner_candidates = list(inner_splitter.split(TRAIN, TRAIN.dx, TRAIN.lesion_id))
# Select the most label-balanced internal fold, using training metadata only.
train_proportions = TRAIN.dx.value_counts(normalize=True).reindex(CLASSES).to_numpy()
valid_candidates = []
for idx, (tr_idx, dv_idx) in enumerate(inner_candidates):
    if set(TRAIN.iloc[tr_idx].dx) != set(CLASSES) or set(TRAIN.iloc[dv_idx].dx) != set(CLASSES):
        continue
    difference = abs(TRAIN.iloc[dv_idx].dx.value_counts(normalize=True)
                     .reindex(CLASSES).to_numpy() - train_proportions).sum()
    valid_candidates.append((difference, idx, tr_idx, dv_idx))
assert valid_candidates, "No seven-class internal development fold"
_, _, tr_idx, internal_idx = min(valid_candidates, key=lambda item: item[0])
INTERNAL_TRAIN = TRAIN.iloc[tr_idx].copy().reset_index(drop=True)
INTERNAL_DEV = TRAIN.iloc[internal_idx].copy().reset_index(drop=True)
assert set(INTERNAL_TRAIN.lesion_id).isdisjoint(INTERNAL_DEV.lesion_id)
assert set(INTERNAL_TRAIN.dx) == set(CLASSES) == set(INTERNAL_DEV.dx)
internal_roles = pd.DataFrame({
    "image_id": TRAIN.image_id,
    "internal_role": np.where(TRAIN.image_id.isin(INTERNAL_DEV.image_id),
                              "internal_dev", "subtrain")})
internal_roles.to_csv(OUT / "tables" / "training_only_internal_split.csv", index=False)
print("Generated split:", partition.split.value_counts().to_dict())
print("Unique lesion groups:", {k: len(v) for k, v in lesion_sets.items()})
print("Training-only internal split:", len(INTERNAL_TRAIN), len(INTERNAL_DEV))
print("Split SHA256:", MANIFEST_SHA256)
print("Test image pixels indexed:", 0)
display(DEV.groupby(["split", "dx"]).size().unstack(fill_value=0).reindex(columns=CLASSES))


## 2. EDA and training-only image-quality analysis (new split)

Explore training class imbalance, distinct lesion groups, missing metadata, age and body-site distributions, example dermoscopic images and a seeded training-image quality sample. The original validation images are reserved for descriptive evaluation and are not included in this EDA.


In [ ]:
train_counts = TRAIN.dx.value_counts().reindex(CLASSES, fill_value=0)
lesion_counts = TRAIN.groupby("dx").lesion_id.nunique().reindex(CLASSES, fill_value=0)
eda = pd.DataFrame({"class": CLASSES, "images": train_counts.to_numpy(),
                    "distinct_lesions": lesion_counts.to_numpy()})
eda["image_share"] = eda.images / len(TRAIN)
eda.to_csv(OUT / "tables" / "training_class_distribution.csv", index=False)
display(eda)

missingness = TRAIN.isna().mean().sort_values(ascending=False).rename("missing_fraction")
missingness.to_csv(OUT / "tables" / "training_missingness.csv")
if "age" in TRAIN:
    age_stats = TRAIN.groupby("dx").age.agg(["count", "median", "mean", "std"])
    age_stats.to_csv(OUT / "tables" / "training_age_by_diagnosis.csv")
if "sex" in TRAIN:
    TRAIN.groupby(["dx", "sex"]).size().unstack(fill_value=0).to_csv(
        OUT / "tables" / "training_sex_by_diagnosis.csv")
if "localization" in TRAIN:
    TRAIN.groupby(["dx", "localization"]).size().to_csv(
        OUT / "tables" / "training_site_by_diagnosis.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(CLASSES, train_counts.to_numpy())
axes[0].set(title="Training images by diagnosis", ylabel="Image count")
axes[1].bar(CLASSES, lesion_counts.to_numpy())
axes[1].set(title="Unique training lesion groups", ylabel="Lesion count")
for ax in axes:
    ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(OUT / "figures" / "eda_training_distribution.png", dpi=160)
plt.show()
plt.close(fig)

if "age" in TRAIN:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(TRAIN.age.dropna().to_numpy(), bins=20)
    ax.set(xlabel="Recorded age (years)", ylabel="Images", title="Training-image age distribution")
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "eda_training_age.png", dpi=160)
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 5))
for col, label in enumerate(CLASSES):
    rows = TRAIN.loc[TRAIN.dx == label].sample(n=2, random_state=42)
    for row_idx, record in enumerate(rows.itertuples()):
        with Image.open(record.path) as source:
            axes[row_idx, col].imshow(source.convert("RGB"))
        axes[row_idx, col].set_title(label if row_idx == 0 else "", fontsize=10)
        axes[row_idx, col].axis("off")
fig.suptitle("Original training images — examples across seven classes", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, .95])
fig.savefig(OUT / "figures" / "eda_training_montage.png", dpi=140)
plt.show()
plt.close(fig)

if CFG.run_quality_audit:
    sample = TRAIN.sample(n=min(400, len(TRAIN)), random_state=CFG.seed)
    checks = []
    for row in sample.itertuples():
        try:
            with Image.open(row.path) as image:
                image.load()
                rgb = np.asarray(image.convert("RGB").resize((64, 64)), dtype=np.float32)
                checks.append({"image_id": row.image_id, "dx": row.dx,
                               "width": image.width, "height": image.height,
                               "brightness": float(rgb.mean()),
                               "contrast": float(rgb.std()), "decode_error": False})
        except (OSError, ValueError) as exc:
            checks.append({"image_id": row.image_id, "dx": row.dx,
                           "width": np.nan, "height": np.nan,
                           "brightness": np.nan, "contrast": np.nan,
                           "decode_error": True})
    quality = pd.DataFrame(checks)
    quality.to_csv(OUT / "tables" / "train_only_image_quality_sample.csv", index=False)
    if quality.decode_error.any():
        raise RuntimeError("Some sampled training images could not be decoded")
    display(quality[["width", "height", "brightness", "contrast"]].describe())

### Optional data-centric ROI ablation

A heuristic lesion-region proposal is implemented as an *optional, independently labeled* data-centric experiment. It is **not** a validated segmentation mask. The default whole-image whole-image training recipe does not crop images. To evaluate this alternative, set `data_policy='wide_roi'` and `approve_roi=True` only after reviewing its QC plot; give the experiment a distinct `output_dir`.


In [ ]:
def propose_wide_roi(image: Image.Image):
    """Return normalized crop coordinates using a color-distance heuristic."""
    try:
        import cv2
    except ImportError as exc:
        raise ImportError("OpenCV is required for the optional ROI ablation") from exc
    rgb = np.asarray(image.convert("RGB"))
    mini = cv2.resize(rgb, (192, 192), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(mini, cv2.COLOR_RGB2LAB).astype(np.float32)
    border = np.concatenate((lab[:12].reshape(-1, 3), lab[-12:].reshape(-1, 3),
                             lab[:, :12].reshape(-1, 3), lab[:, -12:].reshape(-1, 3)))
    skin = np.median(border, axis=0)
    distance = np.linalg.norm(lab - skin, axis=2)
    smooth = cv2.GaussianBlur(np.uint8(np.clip(distance * 2, 0, 255)), (7, 7), 0)
    _, binary = cv2.threshold(smooth, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((11, 11), np.uint8))
    candidates = []
    for contour in cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        x, y, w, h = cv2.boundingRect(contour)
        area = cv2.contourArea(contour) / (192 * 192)
        distance_to_center = np.hypot((x + w / 2) / 192 - .5, (y + h / 2) / 192 - .5)
        if .015 <= area <= .65 and distance_to_center <= .36:
            candidates.append((area - .45 * distance_to_center, (x, y, w, h)))
    if not candidates:
        return (0., 0., 1., 1.), "fallback_full_image"
    x, y, w, h = max(candidates, key=lambda item: item[0])[1]
    x0 = max(0., (x - .2 * w) / 192)
    y0 = max(0., (y - .2 * h) / 192)
    x1 = min(1., (x + 1.2 * w) / 192)
    y1 = min(1., (y + 1.2 * h) / 192)
    if x1 - x0 < .17 or y1 - y0 < .17:
        return (0., 0., 1., 1.), "fallback_full_image"
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    cw, ch = min(1., max(.55, (x1 - x0) * 1.5)), min(1., max(.55, (y1 - y0) * 1.5))
    cx, cy = np.clip(cx, cw / 2, 1 - cw / 2), np.clip(cy, ch / 2, 1 - ch / 2)
    return (float(cx - cw / 2), float(cy - ch / 2),
            float(cx + cw / 2), float(cy + ch / 2)), "heuristic_wide"


if CFG.data_policy == "wide_roi" or CFG.run_roi_audit:
    records = []
    for row in DEV.itertuples():
        with Image.open(row.path) as image:
            box, method = propose_wide_roi(image)
        records.append({"image_id": row.image_id, "roi_method": method,
                        **dict(zip(("x0", "y0", "x1", "y1"), box))})
    roi_table = pd.DataFrame(records)
    assert roi_table.image_id.is_unique and len(roi_table) == len(DEV)
    roi_table.to_csv(OUT / "tables" / "roi_proposals_development.csv", index=False)
    if CFG.data_policy == "wide_roi":
        DEV = DEV.merge(roi_table, on="image_id", validate="one_to_one")
        TRAIN = DEV.loc[DEV.split == "train"].reset_index(drop=True)
        VALID = DEV.loc[DEV.split == "val"].reset_index(drop=True)
        INTERNAL_TRAIN = TRAIN.loc[TRAIN.image_id.isin(INTERNAL_TRAIN.image_id)].reset_index(drop=True)
        INTERNAL_DEV = TRAIN.loc[TRAIN.image_id.isin(INTERNAL_DEV.image_id)].reset_index(drop=True)
    display(roi_table.roi_method.value_counts())
    import matplotlib.patches as patches
    chosen = DEV.groupby("dx", sort=False).head(2).reset_index(drop=True)
    fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 5))
    for idx, row in enumerate(chosen.itertuples()):
        with Image.open(row.path) as source:
            rgb = source.convert("RGB")
        from_roi = roi_table.loc[roi_table.image_id == row.image_id].iloc[0]
        width, height = rgb.size
        x0, y0, x1, y1 = [float(from_roi[c]) for c in ("x0", "y0", "x1", "y1")]
        ax = axes[idx // len(CLASSES), idx % len(CLASSES)]
        ax.imshow(rgb)
        ax.add_patch(patches.Rectangle((x0 * width, y0 * height),
            (x1 - x0) * width, (y1 - y0) * height,
            fill=False, edgecolor="white", linewidth=2))
        ax.set_title(f"{row.dx} / {from_roi.roi_method}", fontsize=8)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "roi_proposal_quality_control.png", dpi=145)
    plt.show()
    plt.close(fig)
    if CFG.data_policy == "wide_roi" and not CFG.approve_roi:
        raise RuntimeError("Review roi_proposal_quality_control.png before enabling approve_roi")

## 3. Data processing and augmentation

Data transforms are deterministic in evaluation mode. In training mode, mild flips, affine transforms and modest color jitter are the default. Optional policies include stronger augmentation, square-root-weighted sampling, and square-root-weighted loss. Every alternative must be run under a distinct configuration and compared using the same train-only model-selection protocol.


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


MEAN = np.array([.485, .456, .406], dtype=np.float32)
STD = np.array([.229, .224, .225], dtype=np.float32)


def build_transform(size: int, training: bool, policy: str):
    ops = [transforms.Resize((size, size))]
    if training:
        if policy == "strong_aug":
            ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                    transforms.RandomAffine(degrees=25, translate=(.08, .08),
                                            scale=(.90, 1.10)),
                    transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.10)]
        else:
            ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                    transforms.RandomAffine(degrees=15, translate=(.04, .04),
                                            scale=(.95, 1.05)),
                    transforms.ColorJitter(brightness=.08, contrast=.08, saturation=.05)]
    return transforms.Compose(ops + [transforms.ToTensor(),
                                     transforms.Normalize(MEAN.tolist(), STD.tolist())])


class LesionImages(Dataset):
    def __init__(self, table: pd.DataFrame, size: int, training: bool, policy: str):
        self.table = table.reset_index(drop=True).copy()
        self.transform = build_transform(size, training, policy)
        self.crop_roi = policy == "wide_roi"
        if self.crop_roi and not {"x0", "y0", "x1", "y1"}.issubset(table.columns):
            raise ValueError("ROI coordinates are missing")

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        with Image.open(row.path) as image:
            image = image.convert("RGB")
            if self.crop_roi:
                width, height = image.size
                coords = tuple(int(v * dim) for v, dim in zip(
                    (row.x0, row.y0, row.x1, row.y1),
                    (width, height, width, height)))
                image = image.crop(coords)
            return self.transform(image), CLASSES.index(row.dx), row.image_id


def training_weights(table: pd.DataFrame):
    counts = table.dx.value_counts().reindex(CLASSES).astype(float)
    assert (counts > 0).all(), "Each training fold must contain every class"
    sample_weights = np.array([1 / np.sqrt(counts[c]) for c in table.dx], dtype=np.float64)
    class_weights = 1 / np.sqrt(counts.to_numpy())
    class_weights /= class_weights.mean()
    return sample_weights, torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)


def make_loader(table, training=False, seed=42, batch_size=None, policy=None):
    policy = policy or CFG.data_policy
    batch_size = batch_size or CFG.batch_size
    sampler = None
    gen = torch.Generator().manual_seed(int(seed))
    if training and policy == "sqrt_sampler":
        weights, _ = training_weights(table)
        sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),
                                        num_samples=len(table), replacement=True, generator=gen)
    return DataLoader(LesionImages(table, CFG.image_size, training, policy),
                      batch_size=batch_size, shuffle=training and sampler is None,
                      sampler=sampler, num_workers=CFG.num_workers,
                      pin_memory=DEVICE.type == "cuda", generator=gen)


# Lightweight unit checks; no images or GPU memory allocation required.
assert build_transform(300, False, "standard") is not None
assert len(LesionImages(TRAIN.head(3), 300, False, "standard")) == 3
assert set(TRAIN.lesion_id).isdisjoint(set(VALID.lesion_id))
print("Data loader and train-only imbalance controls ready")

## 4. EfficientNet-B0 architecture and transfer-learning setup

Initialize from official ImageNet weights, not any previous project checkpoint. A learned 16-dimensional projection feeds both the ordinary seven-class CNN baseline and the Sugeno neuro-fuzzy head. The whole-image fine-tuning configuration uses 300-pixel inputs, two head-only warm-up epochs and fine-tuning of the last four feature blocks.


In [ ]:
class EfficientNetProjector(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        if pretrained:
            if CFG.image_net_weights:
                weights_file = Path(CFG.image_net_weights)
                if not weights_file.is_file():
                    raise FileNotFoundError(f"Missing ImageNet weights: {weights_file}")
                backbone = efficientnet_b0(weights=None)
                backbone.load_state_dict(
                    torch.load(weights_file, map_location="cpu", weights_only=True), strict=True)
            else:
                # Accept the official pretrained EfficientNet-B0 file attached to Kaggle.
                roots = (Path("/kaggle/input"), Path("./data"))
                matches = sorted({p.resolve() for root in roots if root.exists()
                                  for p in root.rglob("*efficientnet_b0*.pth") if p.is_file()})
                if len(matches) == 1:
                    backbone = efficientnet_b0(weights=None)
                    backbone.load_state_dict(
                        torch.load(matches[0], map_location="cpu", weights_only=True), strict=True)
                    print("Loaded official ImageNet pretrained weights:", matches[0])
                elif len(matches) > 1:
                    raise RuntimeError("Multiple ImageNet weight files found; set image_net_weights")
                else:
                    try:
                        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
                    except Exception as exc:
                        raise RuntimeError("Official ImageNet EfficientNet-B0 weights are unavailable. "
                                           "Enable Kaggle Internet or attach the official .pth file") from exc
        else:
            backbone = efficientnet_b0(weights=None)
        self.features = backbone.features
        self.project = nn.Sequential(
            nn.Linear(1280, 128), nn.ReLU(), nn.Dropout(.20),
            nn.Linear(128, 16), nn.LayerNorm(16),
        )
        self.classifier = nn.Linear(16, len(CLASSES))

    def embedding(self, x):
        return self.project(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1))

    def forward(self, x):
        return self.classifier(self.embedding(x))


def set_trainability(model: EfficientNetProjector, depth: int):
    assert 0 <= depth <= len(model.features)
    for parameter in model.features.parameters():
        parameter.requires_grad = False
    if depth:
        for block in list(model.features)[-depth:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
    for parameter in list(model.project.parameters()) + list(model.classifier.parameters()):
        parameter.requires_grad = True


def set_backbone_train_mode(model: EfficientNetProjector, depth: int):
    model.train()
    frozen = list(model.features)[:-depth] if depth else list(model.features)
    for block in frozen:
        block.eval()


def make_cnn(seed: int, pretrained: bool):
    seed_everything(seed)
    model = EfficientNetProjector(pretrained=pretrained).to(DEVICE)
    return model


@torch.no_grad()
def predict_cnn(model: EfficientNetProjector, table: pd.DataFrame, policy=None):
    model.eval()
    truth, logits, names = [], [], []
    loader = make_loader(table, training=False, batch_size=CFG.feature_batch_size, policy=policy)
    for x, y, ids in loader:
        z = model(x.to(DEVICE)).float().cpu().numpy()
        logits.append(z)
        truth.extend(y.numpy().tolist())
        names.extend(ids)
    return np.asarray(truth, dtype=np.int64), np.vstack(logits), names


@torch.no_grad()
def get_embeddings(model: EfficientNetProjector, table: pd.DataFrame, policy=None):
    model.eval()
    vectors, labels, names = [], [], []
    loader = make_loader(table, training=False, batch_size=CFG.feature_batch_size, policy=policy)
    for x, y, ids in loader:
        vectors.append(model.embedding(x.to(DEVICE)).float().cpu().numpy())
        labels.extend(y.numpy().tolist())
        names.extend(ids)
    return np.vstack(vectors).astype(np.float32), np.asarray(labels, dtype=np.int64), names


print("ImageNet-initialized EfficientNet-B0 model and inference functions ready")


## 5. Model evaluation: discrimination, sensitivity, calibration and computational cost

Store raw probabilities and seven-class predictions for every evaluated image. Metrics include accuracy, balanced accuracy, macro/micro/weighted precision, recall and F1; one-versus-rest AUROC and average precision per class; log loss, multiclass Brier score, 15-bin expected calibration error, melanoma false negatives and all confusion-matrix counts. Macro-F1 is the primary predeclared metric; melanoma recall is an important complementary metric.


In [ ]:
def probabilities_from_logits(logits: np.ndarray):
    values = np.asarray(logits, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(CLASSES) or not np.isfinite(values).all():
        raise ValueError("Expected finite, seven-column logits")
    exps = np.exp(values - values.max(axis=1, keepdims=True))
    probabilities = exps / exps.sum(axis=1, keepdims=True)
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-6)
    return probabilities


def evaluate_predictions(truth, logits):
    truth = np.asarray(truth, dtype=int)
    prob = probabilities_from_logits(logits)
    assert len(truth) == len(prob) > 0 and set(truth).issubset(range(len(CLASSES)))
    predicted = prob.argmax(axis=1)
    ids = list(range(len(CLASSES)))
    cm = confusion_matrix(truth, predicted, labels=ids)
    precision = precision_score(truth, predicted, labels=ids, average=None, zero_division=0)
    recall = recall_score(truth, predicted, labels=ids, average=None, zero_division=0)
    f1 = f1_score(truth, predicted, labels=ids, average=None, zero_division=0)
    supports = cm.sum(axis=1)
    specificity = []
    auroc, ap = [], []
    for c in ids:
        tp = int(cm[c, c]); fn = int(cm[c].sum() - tp)
        fp = int(cm[:, c].sum() - tp); tn = int(cm.sum() - tp - fp - fn)
        specificity.append(tn / (tn + fp) if tn + fp else np.nan)
        positive = (truth == c).astype(int)
        if positive.min() == positive.max():
            auroc.append(float("nan")); ap.append(float("nan"))
        else:
            auroc.append(float(roc_auc_score(positive, prob[:, c])))
            ap.append(float(average_precision_score(positive, prob[:, c])))

    confidence = prob.max(axis=1)
    correct = (predicted == truth).astype(float)
    bin_edges = np.linspace(0, 1, 16)
    ece = 0.0
    for index, (left, right) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        selected = ((confidence >= left) &
                    ((confidence <= right) if index == 14 else (confidence < right)))
        if selected.any():
            ece += selected.mean() * abs(correct[selected].mean() - confidence[selected].mean())

    mel = CLASSES.index("mel")
    mel_tp = int(cm[mel, mel]); mel_fn = int(cm[mel].sum() - mel_tp)
    mel_fp = int(cm[:, mel].sum() - mel_tp)
    mel_tn = int(cm.sum() - mel_tp - mel_fn - mel_fp)
    result = {
        "n_images": len(truth),
        "accuracy": float(accuracy_score(truth, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(truth, predicted)),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "weighted_precision": float(precision_score(truth, predicted, labels=ids,
                                                     average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(truth, predicted, labels=ids,
                                               average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(truth, predicted, labels=ids,
                                       average="weighted", zero_division=0)),
        "micro_precision": float(precision_score(truth, predicted, labels=ids,
                                                  average="micro", zero_division=0)),
        "micro_recall": float(recall_score(truth, predicted, labels=ids,
                                            average="micro", zero_division=0)),
        "micro_f1": float(f1_score(truth, predicted, labels=ids,
                                    average="micro", zero_division=0)),
        "macro_auroc_ovr": float(np.mean(auroc)) if np.isfinite(auroc).all() else np.nan,
        "macro_average_precision_ovr": float(np.mean(ap)) if np.isfinite(ap).all() else np.nan,
        "cross_entropy": float(log_loss(truth, prob, labels=ids)),
        "multiclass_brier": float(np.mean(np.square(prob - np.eye(len(ids))[truth]).sum(axis=1))),
        "ece_15bins": float(ece),
        "melanoma_recall": float(recall[mel]),
        "melanoma_precision": float(precision[mel]),
        "melanoma_f1": float(f1[mel]),
        "melanoma_auroc": float(auroc[mel]),
        "melanoma_average_precision": float(ap[mel]),
        "melanoma_true_positive": mel_tp,
        "melanoma_false_negative": mel_fn,
        "melanoma_false_positive": mel_fp,
        "melanoma_true_negative": mel_tn,
    }
    per_class = pd.DataFrame({"class": CLASSES, "support": supports,
                              "precision": precision, "recall": recall, "f1": f1,
                              "specificity": specificity, "auroc_ovr": auroc,
                              "average_precision_ovr": ap})
    return result, per_class, cm, prob


def atomic_json(content, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(content, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)


def atomic_torch_save(content, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(content, tmp)
    os.replace(tmp, path)


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def upsert_result(path, record, key=("configuration", "seed", "split")):
    path = Path(path)
    current = pd.read_csv(path) if path.exists() else pd.DataFrame()
    updated = pd.concat([current, pd.DataFrame([record])], ignore_index=True)
    updated = updated.drop_duplicates(subset=list(key), keep="last")
    temporary = path.with_suffix(".tmp.csv")
    updated.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_model_predictions(label, table, truth, logits, metric_values,
                           inference_seconds=np.nan, training_seconds=np.nan,
                           parameter_count=np.nan, peak_gpu_memory_mb=np.nan,
                           extra=None, split="validation"):
    assert split in {"validation", "cv", "test"}
    assert len(table) == len(truth) == len(logits)
    assert table.image_id.is_unique
    probabilities = probabilities_from_logits(logits)
    pred = probabilities.argmax(axis=1)
    records = pd.DataFrame({"image_id": table.image_id.to_numpy(),
                            "lesion_id": table.lesion_id.to_numpy(),
                            "true": np.asarray(CLASSES)[np.asarray(truth)],
                            "predicted": np.asarray(CLASSES)[pred]})
    for i, name in enumerate(CLASSES):
        records[f"prob_{name}"] = probabilities[:, i]
    records.to_csv(OUT / "tables" / f"{label}_{split}_predictions.csv", index=False)
    row = {"configuration": label, "seed": CFG.seed, "split": split,
           "inference_seconds": inference_seconds,
           "training_seconds": training_seconds,
           "peak_gpu_memory_mb": peak_gpu_memory_mb,
           "parameters": parameter_count, **(extra or {}), **metric_values}
    upsert_result(OUT / "tables" / "results.csv", row)
    return records


# Metric unit tests include both correctly classified and missed minority-class cases.
synthetic_y = np.repeat(np.arange(7), 3)
synthetic_logits = np.full((21, 7), -3.0)
synthetic_logits[np.arange(21), synthetic_y] = 5.0
synthetic_metric, _, synthetic_cm, _ = evaluate_predictions(synthetic_y, synthetic_logits)
assert synthetic_metric["accuracy"] == 1.0
assert synthetic_metric["macro_f1"] == 1.0
assert synthetic_metric["melanoma_false_negative"] == 0
assert np.diag(synthetic_cm).sum() == 21
print("Metric and checkpoint helpers passed unit checks")

## 6. Train-only checkpoint selection and full-training refit

A lesion-disjoint internal subset of the original training partition selects the fine-tuning duration using macro-F1. Then an independently initialized ImageNet CNN is trained on **all original training images** for the selected number of epochs. The original validation partition is evaluated afterward; it never selects the epochs or learning rates. Incremental checkpoint files permit restarting the same run without loading prior E16 models.


In [ ]:
def train_cnn_epochs(model, table, seed, epoch_count, internal_dev=None, stage="selection"):
    """Train for a fixed limit; internal_dev is a training-partition subset only."""
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": list(model.features.parameters()), "lr": CFG.learning_rate_backbone},
        {"params": list(model.project.parameters()) + list(model.classifier.parameters()),
         "lr": CFG.learning_rate_classifier},
    ], weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epoch_count, 1), eta_min=1e-6)
    amp_enabled = DEVICE.type == "cuda" and CFG.use_amp
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    _, weighted_ce = training_weights(table)
    use_weights = CFG.data_policy == "sqrt_weighted"
    best_score = -np.inf
    best_epoch = None
    best_state = None
    stale = 0
    history = []
    t0 = time.perf_counter()
    for epoch in range(epoch_count):
        depth = 0 if epoch < CFG.warmup_epochs else CFG.unfreeze_blocks
        set_trainability(model, depth)
        set_backbone_train_mode(model, depth)
        loader = make_loader(table, training=True, seed=seed + 100 * epoch)
        weighted_loss = 0.0
        seen = 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=amp_enabled):
                logits = model(x)
                loss = F.cross_entropy(logits.float(), y,
                    weight=weighted_ce if use_weights else None,
                    label_smoothing=CFG.label_smoothing)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
            scaler.step(optimizer)
            scaler.update()
            weighted_loss += float(loss.detach()) * len(y)
            seen += len(y)
        record = {"stage": stage, "epoch": epoch + 1,
                  "train_loss": weighted_loss / seen,
                  "backbone_lr": optimizer.param_groups[0]["lr"],
                  "classifier_lr": optimizer.param_groups[1]["lr"],
                  "unfrozen_blocks": depth, "n_train": len(table)}
        if internal_dev is not None:
            assert set(table.lesion_id).isdisjoint(set(internal_dev.lesion_id))
            y_dev, z_dev, _ = predict_cnn(model, internal_dev)
            value = evaluate_predictions(y_dev, z_dev)[0]["macro_f1"]
            record["internal_macro_f1"] = value
            # Warm-up epochs cannot win checkpoint selection.
            if epoch + 1 > CFG.warmup_epochs:
                if value > best_score + 1e-5:
                    best_score, best_epoch, stale = value, epoch + 1, 0
                    best_state = copy.deepcopy({k: v.detach().cpu().clone()
                                                for k, v in model.state_dict().items()})
                else:
                    stale += 1
            print(f"CNN {stage}: epoch {epoch+1}/{epoch_count}, loss={record['train_loss']:.4f}, "
                  f"internal macro-F1={value:.4f}, depth={depth}")
        else:
            print(f"CNN {stage}: epoch {epoch+1}/{epoch_count}, "
                  f"loss={record['train_loss']:.4f}, depth={depth}")
        history.append(record)
        scheduler.step()
        if internal_dev is not None and stale >= CFG.patience:
            break
    if internal_dev is not None:
        if best_state is None:
            raise RuntimeError("No eligible checkpoint obtained after fine-tuning warm-up")
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, best_score, time.perf_counter() - t0


def make_run_fingerprint():
    locked = {"manifest_sha256": MANIFEST_SHA256, "mode": CFG.mode,
              "seed": CFG.seed, "image_size": CFG.image_size,
              "data_policy": CFG.data_policy, "warmup": CFG.warmup_epochs,
              "depth": CFG.unfreeze_blocks, "max_epochs": CFG.max_epochs,
              "lr_backbone": CFG.learning_rate_backbone,
              "lr_classifier": CFG.learning_rate_classifier,
              "learning_rate_anfis": CFG.learning_rate_anfis,
              "weight_decay": CFG.weight_decay, "label_smoothing": CFG.label_smoothing,
              "fuzzy_rules": CFG.fuzzy_rules, "coverage_lambda": CFG.coverage_lambda}
    path = OUT / "run_config_lock.json"
    if path.exists():
        assert json.loads(path.read_text()) == locked, (
            "Configuration changed; use a new output_dir instead of mixing runs")
    else:
        atomic_json(locked, path)
    return locked


RUN_LOCK = make_run_fingerprint()
CNN_PATH = OUT / "checkpoints" / f"CNN_300_deeper_seed{CFG.seed}.pt"
SELECT_CNN_PATH = OUT / "checkpoints" / f"CNN_internal_selected_seed{CFG.seed}.pt"
RESULTS_PATH = OUT / "tables" / "results.csv"
HISTORY_PATH = OUT / "tables" / "cnn_training_history.csv"


def get_or_train_cnn():
    """Train from ImageNet weights or resume a checkpoint created by this run."""
    if CNN_PATH.exists():
        checkpoint = torch.load(CNN_PATH, map_location="cpu", weights_only=True)
        assert checkpoint["manifest_sha256"] == MANIFEST_SHA256
        model = make_cnn(CFG.seed, pretrained=False)
        model.load_state_dict(checkpoint["state_dict"], strict=True)
        return model.eval()

    if DEVICE.type != "cuda":
        warnings.warn("Full CNN fine-tuning on CPU is supported but computationally expensive")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    selected = make_cnn(CFG.seed, pretrained=True)
    selected, h1, best_epoch, best_internal, t1 = train_cnn_epochs(
        selected, INTERNAL_TRAIN, CFG.seed, CFG.max_epochs, INTERNAL_DEV, "selection")
    atomic_torch_save({"state_dict": selected.state_dict(), "manifest_sha256": MANIFEST_SHA256,
                       "selected_epoch": best_epoch, "seed": CFG.seed}, SELECT_CNN_PATH)
    del selected
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    refit = make_cnn(CFG.seed, pretrained=True)
    refit, h2, _, _, t2 = train_cnn_epochs(
        refit, TRAIN, CFG.seed, best_epoch, None, "full_train_refit")
    history = pd.concat((h1, h2), ignore_index=True)
    history.to_csv(HISTORY_PATH, index=False)
    checkpoint = {"state_dict": refit.state_dict(), "manifest_sha256": MANIFEST_SHA256,
                  "selected_epoch": best_epoch, "seed": CFG.seed,
                  "configuration": "CNN_300_deeper", "data_policy": CFG.data_policy}
    atomic_torch_save(checkpoint, CNN_PATH)
    atomic_json({"epoch": best_epoch, "internal_macro_f1": best_internal,
                 "selection_seconds": t1, "refit_seconds": t2},
                OUT / "tables" / "cnn_internal_selection.json")
    return refit.eval()


print("Fresh CNN fine-tuning and internal selection paths ready")


## 7. First-order Sugeno adaptive neuro-fuzzy inference

The 16 Gaussian antecedent prototypes are initialized by training-only class-guided K-means and fitted jointly with first-order linear Sugeno consequents. Firing strengths are normalized with softmax; cross-entropy and an optional marginal-rule-coverage penalty are optimized with AdamW and gradient clipping. A separate training-only internal split selects head duration, followed by a fresh head refit on all original training embeddings. Rule IDs are latent prototypes, not dermatologist-defined attributes.


In [ ]:
class FirstOrderSugeno(nn.Module):
    def __init__(self, centers: np.ndarray, sigma: np.ndarray):
        super().__init__()
        rules, dim = centers.shape
        self.centers = nn.Parameter(torch.as_tensor(centers, dtype=torch.float32).clone())
        self.log_sigma = nn.Parameter(torch.as_tensor(np.log(sigma), dtype=torch.float32).clone())
        self.consequents = nn.Parameter(torch.randn(rules, dim, len(CLASSES)) * .02)
        self.bias = nn.Parameter(torch.zeros(rules, len(CLASSES)))

    def forward(self, inputs, return_firing=False):
        sigma = torch.exp(self.log_sigma).clamp(min=.25, max=10.)
        delta = (inputs[:, None, :] - self.centers[None, :, :]) / sigma[None, :, :]
        firing = torch.softmax(-.5 * delta.square().sum(dim=2), dim=1)
        rule_logits = torch.einsum("bd,rdc->brc", inputs, self.consequents) + self.bias
        logits = torch.sum(firing[:, :, None] * rule_logits, dim=1)
        return (logits, firing) if return_firing else logits


def initialize_rules(x_train, y_train, seed):
    counts = np.bincount(y_train, minlength=len(CLASSES))
    rules_per_class = np.full(len(CLASSES), CFG.fuzzy_rules // len(CLASSES), dtype=int)
    extras = CFG.fuzzy_rules - int(rules_per_class.sum())
    if extras:
        rules_per_class[np.argsort(counts)[:extras]] += 1
    centers, initial_labels = [], []
    for class_index, quantity in enumerate(rules_per_class):
        samples = x_train[y_train == class_index]
        assert len(samples) >= quantity, f"Insufficient samples for {CLASSES[class_index]}"
        cluster = KMeans(n_clusters=int(quantity), random_state=seed, n_init=10).fit(samples)
        centers.extend(cluster.cluster_centers_)
        initial_labels.extend([CLASSES[class_index]] * int(quantity))
    centers = np.asarray(centers, dtype=np.float32)
    sigma = np.broadcast_to(np.maximum(x_train.std(axis=0), .7), centers.shape).copy()
    return centers, sigma.astype(np.float32), initial_labels


@torch.no_grad()
def head_logits(model, features):
    model.eval()
    outputs = []
    for batch in torch.from_numpy(np.asarray(features, dtype=np.float32)).split(1024):
        outputs.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(outputs)


def train_anfis(model, x_train, y_train, epochs, seed, coverage_lambda=0.0,
                x_internal=None, y_internal=None):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate_anfis,
                                  weight_decay=CFG.weight_decay)
    x = torch.from_numpy(np.asarray(x_train, dtype=np.float32)).to(DEVICE)
    y = torch.from_numpy(np.asarray(y_train, dtype=np.int64)).to(DEVICE)
    history, best_score, best_epoch, stale = [], -np.inf, None, 0
    seed_everything(seed)
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y), device=DEVICE)
        losses = []
        for indices in order.split(128):
            optimizer.zero_grad(set_to_none=True)
            logits, firing = model(x[indices], return_firing=True)
            marginal = firing.mean(dim=0)
            penalty = (marginal * len(marginal) - 1).square().mean()
            loss = F.cross_entropy(logits, y[indices]) + coverage_lambda * penalty
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        record = {"epoch": epoch + 1, "train_loss": float(np.mean(losses))}
        if x_internal is not None:
            score = evaluate_predictions(y_internal, head_logits(model, x_internal))[0]["macro_f1"]
            record["internal_macro_f1"] = score
            if score > best_score + 1e-5:
                best_score, best_epoch, stale = score, epoch + 1, 0
            else:
                stale += 1
        history.append(record)
        if x_internal is not None and stale >= CFG.head_patience:
            break
    return model, pd.DataFrame(history), best_epoch, best_score


@torch.no_grad()
def rule_diagnostics(model, features):
    model.eval()
    weights = []
    for batch in torch.from_numpy(np.asarray(features, dtype=np.float32)).split(1024):
        weights.append(model(batch.to(DEVICE), return_firing=True)[1].cpu().numpy())
    f = np.concatenate(weights)
    assert np.allclose(f.sum(axis=1), 1, atol=1e-5)
    effective = np.exp(-(f * np.log(np.clip(f, 1e-12, 1))).sum(axis=1))
    return {"n_images": len(f),
            "mean_effective_rules_per_image": float(effective.mean()),
            "std_effective_rules_across_images": float(effective.std(ddof=1)),
            "median_effective_rules": float(np.median(effective)),
            "fraction_images_with_effective_rules_above_2": float((effective > 2).mean()),
            "rules_with_mean_firing_above_1pct": int((f.mean(axis=0) > .01).sum()),
            "rule_mean_firing": f.mean(axis=0).tolist(),
            "rule_winner_frequency": np.bincount(f.argmax(axis=1), minlength=f.shape[1]).tolist()}


# Shape, normalization and gradient smoke tests.
seed_everything(11)
test_centers = np.random.normal(size=(16, 16)).astype(np.float32)
test_model = FirstOrderSugeno(test_centers, np.ones((16, 16), dtype=np.float32))
test_x = torch.randn(4, 16, requires_grad=True)
test_logits, test_firing = test_model(test_x, return_firing=True)
assert test_logits.shape == (4, 7) and test_firing.shape == (4, 16)
assert torch.allclose(test_firing.sum(1), torch.ones(4), atol=1e-6)
test_logits.sum().backward()
assert test_x.grad is not None and torch.isfinite(test_x.grad).all()
del test_model, test_x, test_logits, test_firing
print("Sugeno membership normalization, output shape and gradients passed")

### Optional training-only data-centric ablation matrix

The default primary recipe uses the standard augmentation policy and no class reweighting. The alternatives below are **independent controlled experiments**, not combined into a single training run. They use identical internal training and development lesion groups, an eight-epoch budget, and the same initialization seed. The outer validation and sealed test are never used to choose a data-centric policy. Set `run_data_policy_ablation=True` in the configuration to execute these additional GPU-intensive experiments. ROI-based training is a separate guarded policy requiring manual crop-quality approval.

In [ ]:
if CFG.run_data_policy_ablation:
    policy_path = OUT / "tables" / "data_policy_internal_ablation.csv"
    for policy in ("standard", "sqrt_sampler", "sqrt_weighted", "strong_aug"):
        # The policy is applied to a fresh CNN; evaluation uses the unaugmented dev images.
        name = f"data_policy_{policy}"
        old_policy = CFG.data_policy
        # No mutation of the frozen config: explicitly pass the policy into the loader.
        def fit_policy(model, train_table, dev_table, policy_name):
            set_trainability(model, 0)
            opt = torch.optim.AdamW([
                {"params": model.features.parameters(), "lr": CFG.learning_rate_backbone},
                {"params": list(model.project.parameters())+list(model.classifier.parameters()),
                 "lr": CFG.learning_rate_classifier}], weight_decay=CFG.weight_decay)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.ablation_epochs)
            _, ce_weights = training_weights(train_table)
            log = []
            for epoch in range(CFG.ablation_epochs):
                depth = 0 if epoch < CFG.warmup_epochs else CFG.unfreeze_blocks
                set_trainability(model, depth)
                set_backbone_train_mode(model, depth)
                losses = []
                for inputs, labels, _ in make_loader(train_table, training=True,
                                                    seed=CFG.seed+epoch, policy=policy_name):
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    opt.zero_grad(set_to_none=True)
                    preds = model(inputs)
                    loss = F.cross_entropy(preds, labels,
                        weight=ce_weights if policy_name == "sqrt_weighted" else None,
                        label_smoothing=CFG.label_smoothing)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    losses.append(float(loss.detach().cpu()))
                sch.step()
                true, predicted_logits, _ = predict_cnn(model, dev_table, policy="standard")
                score = evaluate_predictions(true, predicted_logits)[0]
                log.append({"policy": policy_name, "epoch": epoch+1,
                            "loss": np.mean(losses), "internal_macro_f1": score["macro_f1"],
                            "internal_melanoma_recall": score["melanoma_recall"]})
            return pd.DataFrame(log)
        path = OUT / "tables" / f"{name}_internal_history.csv"
        if path.exists():
            history = pd.read_csv(path)
        else:
            model = make_cnn(CFG.seed, pretrained=True)
            history = fit_policy(model, INTERNAL_TRAIN, INTERNAL_DEV, policy)
            history.to_csv(path, index=False)
            del model
            gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        best_row = history.loc[history.internal_macro_f1.idxmax()].to_dict()
        if policy_path.exists():
            summary = pd.read_csv(policy_path)
            summary = summary[summary.policy != policy]
        else:
            summary = pd.DataFrame()
        pd.concat([summary, pd.DataFrame([best_row])], ignore_index=True).to_csv(
            policy_path, index=False)
    display(pd.read_csv(policy_path).sort_values("internal_macro_f1", ascending=False))
else:
    print("Data-centric training ablations available, but disabled in the primary run")


## 8. Execute from-scratch CNN and Sugeno ANFIS training

Run the whole-image CNN on the fixed original training set, select its training duration on a lesion-disjoint *internal* development split, refit, and calculate descriptive metrics on the original validation set. Extract new 16-dimensional features with this CNN, then train ordinary and coverage-regularized Sugeno heads with train-only normalization, class-guided rule initialization and train-only epoch selection.

The notebook may reuse **only files created in this notebook's own `output_dir`** if the same run is interrupted. It does not import pretrained E16 project checkpoints or embeddings. The matched CNN classifier remains an explicit baseline; no MLP is trained.


In [ ]:
FEATURES_PATH = OUT / "checkpoints" / f"CNN_300_deeper_seed{CFG.seed}_features.npz"
HEAD_SPECS = (("Sugeno_ANFIS", 0.0),
              ("Sugeno_ANFIS_coverage", CFG.coverage_lambda))
HEAD_CHECKPOINTS = {}
HEAD_MODELS = {}

def load_features_file(path):
    with np.load(path, allow_pickle=False) as arr:
        required = {"train_x", "train_y", "train_image_ids", "outer_val_x",
                    "outer_val_y", "outer_val_image_ids"}
        assert required.issubset(arr.files)
        digest_key = ("manifest_sha256" if "manifest_sha256" in arr.files
                      else "source_manifest_sha256")
        assert digest_key in arr.files
        assert str(arr[digest_key].item()) == MANIFEST_SHA256
        assert "cnn_sha256" in arr.files, "Feature file lacks CNN checkpoint provenance"
        assert str(arr["cnn_sha256"].item()) == sha256_file(CNN_PATH), "Stale CNN features detected"
        assert arr["train_x"].shape == (len(TRAIN), 16)
        assert arr["outer_val_x"].shape == (len(VALID), 16)
        assert arr["train_image_ids"].tolist() == TRAIN.image_id.tolist()
        assert arr["outer_val_image_ids"].tolist() == VALID.image_id.tolist()
        assert arr["train_y"].tolist() == [CLASSES.index(c) for c in TRAIN.dx]
        assert arr["outer_val_y"].tolist() == [CLASSES.index(c) for c in VALID.dx]
        assert np.isfinite(arr["train_x"]).all() and np.isfinite(arr["outer_val_x"]).all()
        return tuple(arr[name].copy() for name in
                     ("train_x", "train_y", "outer_val_x", "outer_val_y"))


def try_cached_features():
    """Reuse features only from this run, tied to its exact CNN checkpoint."""
    if not FEATURES_PATH.exists():
        return None
    print("Reusing this run's validated CNN embeddings:", FEATURES_PATH)
    return load_features_file(FEATURES_PATH)


def save_anfis_head(name, model, scaler, selected_epochs, class_labels, init_cnn_name):
    output_path = OUT / "checkpoints" / f"{name}_seed{CFG.seed}.pt"
    atomic_torch_save({"state_dict": model.state_dict(), "seed": CFG.seed,
                       "config": name, "selected_epochs": selected_epochs,
                       "manifest_sha256": MANIFEST_SHA256,
                       "scaler_mean": scaler.mean_.tolist(),
                       "scaler_scale": scaler.scale_.tolist(),
                       "initial_rule_classes": class_labels,
                       "cnn_source": init_cnn_name}, output_path)
    HEAD_CHECKPOINTS[name] = output_path


MODEL = None
X_TRAIN = Y_TRAIN = X_VAL = Y_VAL = None
MODEL = get_or_train_cnn()
start = time.perf_counter()
y_cnn, logits_cnn, ids_cnn = predict_cnn(MODEL, VALID)
assert ids_cnn == VALID.image_id.tolist()
cnn_elapsed = time.perf_counter() - start
cnn_metrics, cnn_per_class, _, _ = evaluate_predictions(y_cnn, logits_cnn)
cnn_label = "CNN_300_deeper"
cnn_timing_path = OUT / "tables" / "cnn_internal_selection.json"
cnn_training_seconds = np.nan
if cnn_timing_path.exists():
    cnn_training_timing = json.loads(cnn_timing_path.read_text())
    cnn_training_seconds = (cnn_training_timing["selection_seconds"]
                            + cnn_training_timing["refit_seconds"])
gpu_peak_mb = (torch.cuda.max_memory_allocated() / 1024**2
               if DEVICE.type == "cuda" else np.nan)
save_model_predictions(cnn_label, VALID, y_cnn, logits_cnn, cnn_metrics,
                       inference_seconds=cnn_elapsed,
                       training_seconds=cnn_training_seconds,
                       parameter_count=sum(p.numel() for p in MODEL.parameters()),
                       peak_gpu_memory_mb=gpu_peak_mb,
                       extra={"image_size": CFG.image_size, "data_policy": CFG.data_policy,
                              "inference_scope": "end_to_end_cnn"})
cnn_per_class.to_csv(OUT / "tables" / f"{cnn_label}_per_class.csv", index=False)
print("CNN descriptive validation: macro-F1", round(cnn_metrics["macro_f1"], 4))

features = try_cached_features()
if features is None:
    started = time.perf_counter()
    x_train, y_train, tr_ids = get_embeddings(MODEL, TRAIN)
    x_val, y_val, va_ids = get_embeddings(MODEL, VALID)
    assert tr_ids == TRAIN.image_id.tolist() and va_ids == VALID.image_id.tolist()
    atomic_npz(FEATURES_PATH, train_x=x_train, train_y=y_train,
               outer_val_x=x_val, outer_val_y=y_val,
               train_image_ids=np.asarray(tr_ids), outer_val_image_ids=np.asarray(va_ids),
               manifest_sha256=np.asarray(MANIFEST_SHA256),
               cnn_sha256=np.asarray(sha256_file(CNN_PATH)))
    features = (x_train, y_train, x_val, y_val)
    print("CNN feature extraction elapsed seconds:",
          round(time.perf_counter() - started, 1))
X_TRAIN, Y_TRAIN, X_VAL, Y_VAL = features
assert X_TRAIN.shape == (len(TRAIN), 16) and X_VAL.shape == (len(VALID), 16)
# Preserve the expensive frozen-feature extraction before fitting ANFIS.
intermediate = (Path("/kaggle/working") if Path("/kaggle/working").exists()
                else OUT.parent) / "ACIT4620_FEATURES_intermediate.zip"
temporary = intermediate.with_suffix(".tmp.zip")
with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (FEATURES_PATH, CNN_PATH):
        archive.write(path, arcname=path.name)
with zipfile.ZipFile(temporary) as archive:
    assert archive.testzip() is None and len(archive.namelist()) == 2
os.replace(temporary, intermediate)
print("Intermediate CNN/feature backup ready:", intermediate)

for name, regularizer in HEAD_SPECS:
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    head_path = OUT / "checkpoints" / f"{name}_seed{CFG.seed}.pt"
    selected_epoch = None
    assert SELECT_CNN_PATH.exists()
    candidate = torch.load(SELECT_CNN_PATH, map_location="cpu", weights_only=True)
    selection_cnn = make_cnn(CFG.seed, pretrained=False)
    selection_cnn.load_state_dict(candidate["state_dict"], strict=True)
    sel_train_x, sel_train_y, sel_tr_ids = get_embeddings(selection_cnn, INTERNAL_TRAIN)
    sel_dev_x, sel_dev_y, sel_dev_ids = get_embeddings(selection_cnn, INTERNAL_DEV)
    assert sel_tr_ids == INTERNAL_TRAIN.image_id.tolist()
    assert sel_dev_ids == INTERNAL_DEV.image_id.tolist()
    del selection_cnn
    gc.collect()
    sel_scaler = StandardScaler().fit(sel_train_x)
    xtr = sel_scaler.transform(sel_train_x).astype(np.float32)
    xdv = sel_scaler.transform(sel_dev_x).astype(np.float32)
    seed_everything(CFG.seed)
    centers, sigmas, _ = initialize_rules(xtr, sel_train_y, CFG.seed)
    temp_head = FirstOrderSugeno(centers, sigmas)
    temp_head, head_history, best_epoch, best_score = train_anfis(
        temp_head, xtr, sel_train_y, CFG.head_max_epochs, CFG.seed,
        regularizer, xdv, sel_dev_y)
    assert best_epoch is not None
    selected_epoch = best_epoch
    head_history.to_csv(OUT / "tables" / f"{name}_internal_selection.csv", index=False)
    del temp_head
    gc.collect()
    atomic_json({"best_epoch": selected_epoch, "internal_macro_f1": best_score},
                OUT / "tables" / f"{name}_internal_selection.json")
    if head_path.exists():
        saved = torch.load(head_path, map_location="cpu", weights_only=True)
        assert saved["manifest_sha256"] == MANIFEST_SHA256
        scaler = StandardScaler()
        scaler.mean_ = np.asarray(saved["scaler_mean"])
        scaler.scale_ = np.asarray(saved["scaler_scale"])
        scaler.n_features_in_ = X_TRAIN.shape[1]
        x_train_scaled = scaler.transform(X_TRAIN).astype(np.float32)
        centers = saved["state_dict"]["centers"].numpy()
        sigma = np.exp(saved["state_dict"]["log_sigma"].numpy())
        head = FirstOrderSugeno(centers, sigma).to(DEVICE)
        head.load_state_dict(saved["state_dict"])
        initial_labels = saved["initial_rule_classes"]
        selected_epoch = saved["selected_epochs"]
    else:
        scaler = StandardScaler().fit(X_TRAIN)
        x_train_scaled = scaler.transform(X_TRAIN).astype(np.float32)
        seed_everything(CFG.seed)
        centers, sigma, initial_labels = initialize_rules(x_train_scaled, Y_TRAIN, CFG.seed)
        head = FirstOrderSugeno(centers, sigma)
        t0 = time.perf_counter()
        head, history, _, _ = train_anfis(head, x_train_scaled, Y_TRAIN,
                                           selected_epoch, CFG.seed, regularizer)
        history.to_csv(OUT / "tables" / f"{name}_training_history.csv", index=False)
        atomic_json({"training_seconds": time.perf_counter() - t0},
                    OUT / "tables" / f"{name}_timing.json")
        save_anfis_head(name, head, scaler, selected_epoch, initial_labels,
                        "CNN_300_deeper")
    x_val_scaled = scaler.transform(X_VAL).astype(np.float32)
    head = head.to(DEVICE).eval()
    t0 = time.perf_counter()
    logits = head_logits(head, x_val_scaled)
    elapsed = time.perf_counter() - t0
    metrics, per_class, _, _ = evaluate_predictions(Y_VAL, logits)
    head_time_path = OUT / "tables" / f"{name}_timing.json"
    head_training_seconds = (json.loads(head_time_path.read_text()).get("training_seconds", np.nan)
                             if head_time_path.exists() else np.nan)
    save_model_predictions(name, VALID, Y_VAL, logits, metrics,
                           inference_seconds=elapsed,
                           training_seconds=head_training_seconds,
                           parameter_count=sum(p.numel() for p in head.parameters()),
                           peak_gpu_memory_mb=(torch.cuda.max_memory_allocated()/1024**2
                                               if DEVICE.type == "cuda" else np.nan),
                           extra={"selected_head_epochs": selected_epoch,
                                  "regularization_lambda": regularizer,
                                  "inference_scope": "head_only_precomputed_cnn_features"})
    per_class.to_csv(OUT / "tables" / f"{name}_per_class.csv", index=False)
    diagnostic = rule_diagnostics(head, x_val_scaled)
    diagnostic.update({"configuration": name,
                       "initial_rule_classes": initial_labels})
    atomic_json(diagnostic, OUT / "tables" / f"{name}_rule_diagnostics.json")
    HEAD_MODELS[name] = (head, scaler)
    print(f"{name}: macro-F1={metrics['macro_f1']:.4f}; "
          f"melanoma recall={metrics['melanoma_recall']:.4f}; "
          f"effective rules={diagnostic['mean_effective_rules_per_image']:.3f}")
print("Model and head evaluation complete; new test image pixels still untouched")


## 9. Validation figures and calibrated probability analysis

Recompute metrics from this run's saved prediction probabilities, then produce count-normalized confusion matrices, per-class ROC and precision–recall curves, calibration plots and report-ready per-class tables for CNN, Sugeno ANFIS and coverage-regularized ANFIS.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve


def metrics_from_saved_predictions(csv_path):
    frame = pd.read_csv(csv_path)
    required = {"image_id", "lesion_id", "true", "predicted"} | {f"prob_{c}" for c in CLASSES}
    assert required.issubset(frame.columns)
    truth = np.array([CLASSES.index(value) for value in frame.true])
    probability = frame[[f"prob_{c}" for c in CLASSES]].to_numpy()
    assert np.isfinite(probability).all() and np.allclose(probability.sum(1), 1, atol=1e-4)
    # Logarithms of saved probabilities recover equivalent logits.
    metrics, per_class, cm, _ = evaluate_predictions(truth, np.log(np.clip(probability, 1e-15, 1)))
    assert np.array_equal(np.asarray(CLASSES)[probability.argmax(1)], frame.predicted.to_numpy())
    return frame, metrics, per_class, cm


def plot_metrics(label, split="validation"):
    """Regenerate validated diagnostic figures from saved per-image probabilities."""
    assert split in {"validation", "test"}
    path = OUT / "tables" / f"{label}_{split}_predictions.csv"
    if not path.exists():
        return
    frame, metric_values, per_class, matrix = metrics_from_saved_predictions(path)
    cm_norm = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    panels = ((axes[0], matrix, "d", "Counts", None),
              (axes[1], cm_norm, ".2f", "Row-normalized", 1.0))
    for ax, data, fmt, title, vmax in panels:
        view = ax.imshow(data, cmap="Blues", vmin=0, vmax=vmax,
                         interpolation="nearest")
        ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
               xticklabels=CLASSES, yticklabels=CLASSES,
               xlabel="Predicted class", ylabel="Actual class",
               title=f"{label}: {title}")
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
        for i in range(len(CLASSES)):
            for j in range(len(CLASSES)):
                val = data[i, j]
                color = "white" if view.norm(val) >= .55 else "black"
                ax.text(j, i, format(val, fmt), ha="center", va="center",
                        fontsize=8, color=color)
        fig.colorbar(view, ax=ax, fraction=.046, pad=.04)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / f"{label}_{split}_confusion_matrices.png",
                dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    y = np.array([CLASSES.index(c) for c in frame.true])
    prob = frame[[f"prob_{c}" for c in CLASSES]].to_numpy()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for idx, label_name in enumerate(CLASSES):
        positive = (y == idx).astype(int)
        if positive.min() == positive.max():
            continue
        fpr, tpr, _ = roc_curve(positive, prob[:, idx])
        rec = precision_recall_curve(positive, prob[:, idx])
        axes[0].plot(fpr, tpr, label=f"{label_name}: {roc_auc_score(positive, prob[:,idx]):.2f}")
        axes[1].plot(rec[1], rec[0],
                     label=f"{label_name}: {average_precision_score(positive, prob[:,idx]):.2f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", alpha=.4)
    axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="One-vs-rest ROC")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="One-vs-rest precision–recall")
    for ax in axes:
        ax.legend(fontsize=7)
        ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / f"{label}_{split}_roc_pr.png", dpi=160)
    plt.show()
    plt.close(fig)

    confidence, predicted = prob.max(axis=1), prob.argmax(axis=1)
    edges = np.linspace(0, 1, 16)
    mean_confidence, mean_accuracy = [], []
    for i, (left, right) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (confidence >= left) & ((confidence <= right) if i == 14 else (confidence < right))
        if mask.any():
            mean_confidence.append(float(confidence[mask].mean()))
            mean_accuracy.append(float((predicted[mask] == y[mask]).mean()))
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    ax.plot(mean_confidence, mean_accuracy, "o-", label="Observed confidence")
    ax.set(xlabel="Mean confidence", ylabel="Observed accuracy",
           title=f"{label} | ECE={metric_values['ece_15bins']:.3f}")
    ax.legend()
    ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / f"{label}_{split}_reliability.png", dpi=160)
    plt.show()
    plt.close(fig)
    per_class.to_csv(OUT / "tables" / f"{label}_{split}_per_class_recomputed.csv", index=False)


for _model_label in ("CNN_300_deeper", "Sugeno_ANFIS", "Sugeno_ANFIS_coverage"):
    plot_metrics(_model_label)

if RESULTS_PATH.exists():
    scoreboard = pd.read_csv(RESULTS_PATH)
    metrics_to_show = ["configuration", "accuracy", "balanced_accuracy", "macro_f1",
                       "macro_precision", "macro_recall", "weighted_f1",
                       "macro_auroc_ovr", "macro_average_precision_ovr",
                       "melanoma_recall", "melanoma_precision", "ece_15bins"]
    display(scoreboard[[col for col in metrics_to_show if col in scoreboard.columns]])


### Descriptive 95% lesion-group bootstrap intervals

Resample lesion IDs (not individual images) to describe uncertainty in the newly generated validation partition. These intervals are not equivalent to independently validated performance or uncertainty across new hospitals and devices.


In [ ]:
def grouped_bootstrap(frame, repeats=300, seed=42):
    if repeats == 0:
        return pd.DataFrame()
    rng = np.random.default_rng(seed)
    y = np.array([CLASSES.index(c) for c in frame.true])
    prob = frame[[f"prob_{name}" for name in CLASSES]].to_numpy()
    predicted = prob.argmax(axis=1)
    groups = frame.lesion_id.to_numpy()
    distinct = np.unique(groups)
    lookup = {group: np.flatnonzero(groups == group) for group in distinct}
    rows = []
    for _ in range(repeats):
        sampled = rng.choice(distinct, size=len(distinct), replace=True)
        ix = np.concatenate([lookup[group] for group in sampled])
        valid = y[ix]
        guess = predicted[ix]
        rows.append({"macro_f1": f1_score(valid, guess, labels=range(7),
                                          average="macro", zero_division=0),
                     "accuracy": accuracy_score(valid, guess),
                     "melanoma_recall": recall_score(valid == CLASSES.index("mel"),
                                                    guess == CLASSES.index("mel"), zero_division=0)})
    sample = pd.DataFrame(rows)
    return pd.DataFrame({"metric": sample.columns,
                         "lower_2.5pct": sample.quantile(.025).values,
                         "upper_97.5pct": sample.quantile(.975).values})


if CFG.bootstrap_repeats:
    for name in ("CNN_300_deeper", "Sugeno_ANFIS", "Sugeno_ANFIS_coverage"):
        path = OUT / "tables" / f"{name}_validation_predictions.csv"
        if path.exists():
            ci = grouped_bootstrap(pd.read_csv(path), CFG.bootstrap_repeats, CFG.seed)
            ci.insert(0, "configuration", name)
            ci.to_csv(OUT / "tables" / f"{name}_group_bootstrap_95pct.csv", index=False)


## 10. End-to-end, training-only three-fold cross-validation

This section is **enabled by default**. Each fold trains an independently initialized ImageNet CNN on that fold's training lesions for 16 fixed epochs, then extracts fold-specific embeddings and fits new Sugeno heads and train-only normalizers. No original validation or sealed-test images enter fold training or evaluation. The entire fold outputs are saved incrementally, including fold-level probabilities and mean ± sample standard deviation.

Three folds report variability between training/evaluation partitions, not repeated-seed uncertainty. Each fold uses prespecified durations without selecting CNN checkpoints on its evaluation fold. Fresh CV requires substantial GPU time and may need a separate session; `run_cv=False` explicitly creates an incomplete pilot instead.


In [ ]:
def backup(stage="LATEST"):
    """Create and validate separate report/checkpoint archives with atomic replacement."""
    root = Path("/kaggle/working") if Path("/kaggle/working").exists() else OUT.parent
    outputs = []
    for suffix, extensions in (("reports", {".csv", ".json", ".png", ".md"}),
                               ("checkpoints", {".pt", ".npz"})):
        destination = root / f"ACIT4620_HybridAI_{stage}_{suffix}.zip"
        temp = destination.with_suffix(".tmp.zip")
        files = sorted(p for p in OUT.rglob("*") if p.is_file() and p.suffix in extensions)
        with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED,
                             compresslevel=5) as z:
            for path in files:
                z.write(path, arcname=path.relative_to(OUT))
        with zipfile.ZipFile(temp) as z:
            assert z.testzip() is None
            assert len(z.namelist()) == len(files)
        os.replace(temp, destination)
        outputs.append(destination)
        print(destination.name, "| files:", len(files), "| MB:",
              round(destination.stat().st_size / 1024 ** 2, 2))
    return outputs


backup("PRE_CV")


if CFG.run_cv:
    folds = StratifiedGroupKFold(n_splits=CFG.cv_folds, shuffle=True, random_state=42)
    summary_file = OUT / "cv" / "end_to_end_cv_results.csv"
    for fold, (train_idx, held_idx) in enumerate(
            folds.split(TRAIN, TRAIN.dx, TRAIN.lesion_id), start=1):
        fold_train = TRAIN.iloc[train_idx].reset_index(drop=True)
        fold_eval = TRAIN.iloc[held_idx].reset_index(drop=True)
        assert set(fold_train.lesion_id).isdisjoint(fold_eval.lesion_id)
        assert set(fold_train.dx) == set(fold_eval.dx) == set(CLASSES)
        seed = CFG.seed + fold
        names = ("CNN", "Sugeno_ANFIS", "Sugeno_ANFIS_coverage")
        fold_oof_file = OUT / "cv" / f"fold_{fold}_oof_predictions.csv"
        if summary_file.exists() and fold_oof_file.exists():
            current = pd.read_csv(summary_file)
            completed = set(current.loc[current.fold == fold, "configuration"])
            if set(names).issubset(completed):
                print("Skipping fully recorded fold", fold)
                continue
        fold_oof = []
        fold_cnn = make_cnn(seed, pretrained=True)
        fold_cnn, fold_history, _, _, train_seconds = train_cnn_epochs(
            fold_cnn, fold_train, seed, CFG.cv_cnn_epochs,
            internal_dev=None, stage=f"cv_fold_{fold}")
        fold_history.to_csv(OUT / "cv" / f"fold_{fold}_cnn_history.csv", index=False)
        y_eval, cnn_logits, ids_eval = predict_cnn(fold_cnn, fold_eval)
        assert ids_eval == fold_eval.image_id.tolist()
        cnn_metrics, _, _, cnn_prob = evaluate_predictions(y_eval, cnn_logits)
        upsert_result(summary_file, {"fold": fold, "configuration": "CNN",
                     "n_train": len(fold_train), "n_eval": len(fold_eval), "seed": seed,
                     "training_seconds": train_seconds, **cnn_metrics},
                     key=("configuration", "fold"))
        fold_x_train, fold_y_train, _ = get_embeddings(fold_cnn, fold_train)
        fold_x_eval, fold_y_eval, _ = get_embeddings(fold_cnn, fold_eval)
        assert np.array_equal(fold_y_eval, y_eval)
        # Fitted strictly on fold_train, including normalizer and class prototypes.
        fold_scaler = StandardScaler().fit(fold_x_train)
        tx = fold_scaler.transform(fold_x_train).astype(np.float32)
        vx = fold_scaler.transform(fold_x_eval).astype(np.float32)
        for head_name, reg in HEAD_SPECS:
            seed_everything(seed)
            centers, sigma, _ = initialize_rules(tx, fold_y_train, seed)
            fold_head = FirstOrderSugeno(centers, sigma)
            fold_head, head_history, _, _ = train_anfis(
                fold_head, tx, fold_y_train, CFG.cv_anfis_epochs, seed, reg)
            head_history.to_csv(OUT / "cv" / f"fold_{fold}_{head_name}_training.csv", index=False)
            head_logits_fold = head_logits(fold_head, vx)
            values, _, _, probabilities = evaluate_predictions(y_eval, head_logits_fold)
            upsert_result(summary_file, {"fold": fold, "configuration": head_name,
                         "n_train": len(fold_train), "n_eval": len(fold_eval),
                         "seed": seed, **values}, key=("configuration", "fold"))
            diagnostic = rule_diagnostics(fold_head.to(DEVICE), vx)
            atomic_json(diagnostic, OUT / "cv" / f"fold_{fold}_{head_name}_rules.json")
            for row, p in zip(fold_eval.itertuples(), probabilities):
                fold_oof.append({"fold": fold, "configuration": head_name,
                                "image_id": row.image_id, "lesion_id": row.lesion_id,
                                "given_dx": row.dx, **{f"prob_{c}": float(p[j])
                                                    for j, c in enumerate(CLASSES)}})
            del fold_head
        for row, p in zip(fold_eval.itertuples(), cnn_prob):
            fold_oof.append({"fold": fold, "configuration": "CNN",
                            "image_id": row.image_id, "lesion_id": row.lesion_id,
                            "given_dx": row.dx,
                            **{f"prob_{c}": float(p[j]) for j, c in enumerate(CLASSES)}})
        pd.DataFrame(fold_oof).to_csv(fold_oof_file, index=False)
        del fold_cnn
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        backup("CV_LATEST")

    cv_results = pd.read_csv(summary_file)
    expected = {(fold, name) for fold in range(1, CFG.cv_folds + 1)
                for name in ("CNN", *[name for name, _ in HEAD_SPECS])}
    assert set(zip(cv_results.fold, cv_results.configuration)) == expected
    metric_columns = ["macro_f1", "accuracy", "balanced_accuracy", "melanoma_recall",
                      "macro_auroc_ovr", "macro_average_precision_ovr"]
    cv_summary = cv_results.groupby("configuration")[metric_columns].agg(["mean", "std", "count"])
    cv_summary.to_csv(OUT / "cv" / "cv_mean_std.csv")
    display(cv_summary)

    # Out-of-fold probabilities support only a label-quality *review* queue.
    # No labels are changed or removed automatically.
    fold_files = [OUT / "cv" / f"fold_{f}_oof_predictions.csv"
                  for f in range(1, CFG.cv_folds + 1)]
    assert all(p.exists() for p in fold_files), "A fold prediction file is missing"
    oof = pd.concat([pd.read_csv(p) for p in fold_files], ignore_index=True)
    assert len(oof) == CFG.cv_folds * len(TRAIN), "Incomplete three-model OOF prediction coverage"
    assert not oof.duplicated(subset=["configuration", "image_id"]).any()
    oof.to_csv(OUT / "cv" / "train_only_oof_probabilities.csv", index=False)
    oof["given_label_probability"] = [getattr(row, f"prob_{row.given_dx}")
                                       for row in oof.itertuples(index=False)]
    oof.sort_values(["configuration", "given_label_probability"]).to_csv(
        OUT / "cv" / "train_only_label_review_queue.csv", index=False)
    # Optional Cleanlab ranking from genuine training-only out-of-fold probabilities.
    # An annotation review is required; the pipeline never relabels or deletes samples.
    if CFG.cleanlab_review:
        try:
            from cleanlab.filter import find_label_issues
        except ImportError:
            print("Cleanlab not installed; OOF confidence ranking was still saved")
        else:
            cnn_oof = oof.loc[oof.configuration == "CNN"].sort_values("image_id")
            if len(cnn_oof) == len(TRAIN) and cnn_oof.image_id.is_unique:
                probabilities = cnn_oof[[f"prob_{c}" for c in CLASSES]].to_numpy()
                given = np.array([CLASSES.index(c) for c in cnn_oof.given_dx])
                ranked = find_label_issues(given, probabilities,
                                          return_indices_ranked_by="self_confidence")
                review = cnn_oof.iloc[ranked].copy()
                review.head(max(1, int(.05 * len(review)))).to_csv(
                    OUT / "cv" / "training_only_cleanlab_review_top5pct.csv", index=False)
else:
    print("Grouped CV disabled by configuration; report this run as an incomplete pilot")


### 10A. Cross-validation and fuzzy-rule diagnostics

Generate report-ready plots from **this run's** three-fold training-only CV and validation rule diagnostics. No retraining is performed. Blue confusion matrices are already produced above. Cross-validation bars show mean ± sample SD and must not be interpreted as confidence intervals or independent external validation.


In [ ]:
# Reusable visualizations for training-only CV and rule activation.
def export_cv_and_rule_figures():
    cv_path = OUT / "cv" / "end_to_end_cv_results.csv"
    assert cv_path.exists(), "Three-fold CV must finish before plotting"
    cv = pd.read_csv(cv_path)
    expected_models = ("CNN", "Sugeno_ANFIS", "Sugeno_ANFIS_coverage")
    required = {"fold", "configuration", "macro_f1", "balanced_accuracy", "melanoma_recall"}
    assert required.issubset(cv.columns)
    assert len(cv) == 3 * len(expected_models)
    assert not cv.duplicated(["fold", "configuration"]).any()
    assert all(set(cv.loc[cv.configuration == model, "fold"]) == {1, 2, 3}
               for model in expected_models)

    colors = ("#123f73", "#468abd", "#9dc7e5")
    metrics = (("macro_f1", "Macro-F1"),
               ("balanced_accuracy", "Balanced accuracy"),
               ("melanoma_recall", "Melanoma recall"))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax, (column, title) in zip(axes, metrics):
        model_mean = cv.groupby("configuration")[column].mean().reindex(expected_models)
        model_sd = cv.groupby("configuration")[column].std().reindex(expected_models)
        x = np.arange(len(expected_models))
        ax.bar(x, model_mean.to_numpy(), yerr=model_sd.to_numpy(),
               capsize=5, color=colors, edgecolor="#1e293b", linewidth=.5)
        ax.set_xticks(x, ("CNN", "ANFIS", "Coverage"))
        ax.set_ylim(0, min(1., float((model_mean + model_sd).max()) + .12))
        ax.set_ylabel(title)
        ax.set_title(f"3-fold CV: {title} (mean ± SD)")
        ax.grid(axis="y", alpha=.20)
        ax.set_axisbelow(True)
        for i, val in enumerate(model_mean):
            ax.text(i, val + model_sd.iloc[i] + .017, f"{val:.3f}",
                    ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "cv_mean_sd_model_comparison.png",
                dpi=200, bbox_inches="tight")
    plt.show(); plt.close(fig)

    cv_wide = cv.pivot(index="fold", columns="configuration", values="macro_f1")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for model, color in zip(expected_models, colors):
        ax.plot(cv_wide.index, cv_wide[model], marker="o", linewidth=2,
                color=color, label=("CNN" if model == "CNN" else model))
    ax.set(xticks=[1, 2, 3], xlabel="Held-out fold", ylabel="Macro-F1",
           title="Paired training-only cross-validation folds")
    ax.grid(alpha=.2); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "cv_fold_macro_f1.png",
                dpi=200, bbox_inches="tight")
    plt.show(); plt.close(fig)

    pairwise = cv_wide.copy()
    pairwise["ANFIS_minus_CNN"] = pairwise["Sugeno_ANFIS"] - pairwise["CNN"]
    pairwise["Coverage_minus_CNN"] = pairwise["Sugeno_ANFIS_coverage"] - pairwise["CNN"]
    pairwise.to_csv(OUT / "cv" / "paired_fold_macro_f1_differences.csv")

    heads = ("Sugeno_ANFIS", "Sugeno_ANFIS_coverage")
    diagnostics = []
    for head in heads:
        path = OUT / "tables" / f"{head}_rule_diagnostics.json"
        assert path.exists(), f"Missing neuro-fuzzy diagnostics: {path}"
        data = json.loads(path.read_text())
        means = np.asarray(data["rule_mean_firing"], dtype=float)
        winner_counts = np.asarray(data["rule_winner_frequency"], dtype=float)
        assert means.shape == winner_counts.shape == (CFG.fuzzy_rules,)
        assert np.isclose(means.sum(), 1, atol=1e-4)
        assert int(winner_counts.sum()) == int(data["n_images"])
        diagnostics.append((means, winner_counts / winner_counts.sum(), data))

    fig, axes = plt.subplots(2, 1, figsize=(13, 5.1), sharex=True)
    titles = ("Mean normalized firing strength", "Winner frequency (fraction of images)")
    for ax, key, title in zip(axes, (0, 1), titles):
        values = np.vstack([row[key] for row in diagnostics])
        image = ax.imshow(values, cmap="Blues", aspect="auto", vmin=0)
        ax.set_yticks((0, 1), ("Standard ANFIS", "Coverage ANFIS"))
        ax.set_title(title)
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                val = float(values[i, j])
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6.5,
                        color=("white" if image.norm(val) > .55 else "black"))
        fig.colorbar(image, ax=ax, fraction=.025, pad=.02)
    axes[-1].set_xticks(range(CFG.fuzzy_rules),
                        [f"R{i+1}" for i in range(CFG.fuzzy_rules)])
    axes[-1].set_xlabel("Fuzzy rule (prototype index)")
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "anfis_rule_usage_heatmaps.png",
                dpi=200, bbox_inches="tight")
    plt.show(); plt.close(fig)

    summary = pd.DataFrame([{"configuration": name,
                             "effective_rules_per_image": data["mean_effective_rules_per_image"],
                             "active_rules_over_1pct": data["rules_with_mean_firing_above_1pct"],
                             "image_count": data["n_images"]}
                            for name, (_, _, data) in zip(heads, diagnostics)])
    summary.to_csv(OUT / "tables" / "rule_usage_model_comparison.csv", index=False)
    display(summary)
    return summary

if CFG.run_cv:
    rule_comparison = export_cv_and_rule_figures()


### Run outputs and independent-source boundary

All saved results and plots in this notebook are generated from **this run only**. The run uses only the raw metadata, images, pretrained backbone initialization, and its own generated artifacts. Published methods cited in the report should be verified separately; this notebook documents the executable experimental protocol and its output provenance.


In [ ]:
print("From-scratch run: no external results or historical checkpoints are imported.")


### One-time locked test-set evaluation (predeclared primary model) (after model choice)

For a final study, select **one model and its complete protocol before unlocking** the newly generated test split. The final protocol is `final_test_once=False`. To enable this step, set `final_test_once=True` in the top configuration on a fresh run or an unchanged configuration session. Do not select architecture, epoch count, thresholds or hyperparameters from the test results. This section creates test image paths only after the explicit unlock and records that the test was evaluated.

In [ ]:
if CFG.final_test_once:
    # Do not permit a second report from changed models or hyperparameters.
    marker = OUT / "tables" / "sealed_test_evaluated.json"
    assert not marker.exists(), "Test already evaluated in this output directory"
    assert CFG.run_cv, "Complete grouped CV before final test evaluation"
    assert RESULTS_PATH.exists() and (OUT / "cv" / "cv_mean_std.csv").exists()
    model_choice = CFG.final_test_model
    assert model_choice == "Sugeno_ANFIS_coverage", (
        "Primary final-test model differs from the predeclared coverage hybrid")
    atomic_json({"selected_model": model_choice,
                 "primary_selection_metric": "three-fold training-only CV macro-F1",
                 "selected_before_test_pixels": True,
                 "test_used_for_training_or_selection": False,
                 "dataset_partition_sha256": MANIFEST_SHA256},
                OUT / "tables" / "final_test_model_selection.json")
    # The test split was assigned from labels; image paths are first accessed here.
    chosen_ids = set(TEST_IDS.tolist())
    canonical_files = {}
    for folder in image_dirs:
        for path in folder.glob("*.jpg"):
            if path.stem in chosen_ids:
                if path.stem in canonical_files:
                    raise RuntimeError("Duplicate sealed-test image")
                canonical_files[path.stem] = str(path)
    assert set(canonical_files) == chosen_ids
    heldout_table = partition.loc[partition.split == "test"].merge(
        metadata_raw.drop(columns="lesion_id"), on="image_id", validate="one_to_one")
    heldout_table = heldout_table.reset_index(drop=True)
    heldout_table["path"] = heldout_table.image_id.map(canonical_files)
    if CFG.data_policy == "wide_roi":
        # The ROI policy is deterministic and does not use test diagnoses.
        coords = []
        for row in heldout_table.itertuples():
            with Image.open(row.path) as im:
                box, _ = propose_wide_roi(im)
            coords.append(box)
        heldout_table[["x0", "y0", "x1", "y1"]] = coords
    if model_choice == "CNN_300_deeper":
        labels, scores, _ = predict_cnn(MODEL, heldout_table)
    else:
        selected_head, selected_scaler = HEAD_MODELS[model_choice]
        tensor, labels, _ = get_embeddings(MODEL, heldout_table)
        scaled = selected_scaler.transform(tensor).astype(np.float32)
        scores = head_logits(selected_head, scaled)
    score_values, class_values, _, _ = evaluate_predictions(labels, scores)
    records = save_model_predictions(model_choice, heldout_table, labels, scores,
                                     score_values, split="test")
    class_values.to_csv(OUT / "tables" / "final_test_per_class.csv", index=False)
    atomic_json({"test_evaluated": True, "model": model_choice,
                 "split_sha256": MANIFEST_SHA256,
                 "evaluation_timestamp_utc": pd.Timestamp.utcnow().isoformat()}, marker)
    print("One-time final test results", model_choice)
    display(pd.DataFrame([score_values]))
    # Reuse the same diagnostic plots, now explicitly labeled as held-out test.
    plot_metrics(model_choice, split="test")
else:
    print("Sealed test remains locked; zero test image pixels have been loaded")


## 11. Explainable AI: CNN and CNN–ANFIS Grad-CAM

For seven examples from the original validation partition, calculate Grad-CAM separately for the CNN logits and the differentiable CNN→Sugeno ANFIS classification path, including the train-only feature scaler. The shared backbone, entire trained classifier path and predicted-class logits define the explanations. Maps show model sensitivity, **not** diagnostic validation. Column headings are shown only once to avoid label overlap.


In [ ]:
def gradcam_for_prediction(cnn, image, head=None, scaler=None):
    activations = []

    def capture(_module, _inputs, output):
        output.retain_grad()
        activations.append(output)

    handle = cnn.features[-1].register_forward_hook(capture)
    cnn.zero_grad(set_to_none=True)
    if head is not None:
        head.zero_grad(set_to_none=True)
    try:
        with torch.enable_grad():
            feature = cnn.embedding(image)
            if head is None:
                logits = cnn.classifier(feature)
            else:
                mu = torch.tensor(scaler.mean_, dtype=feature.dtype, device=feature.device)
                sd = torch.tensor(scaler.scale_, dtype=feature.dtype, device=feature.device)
                logits = head((feature - mu) / sd)
            pred = int(logits.argmax(dim=1).item())
            logits[0, pred].backward()
        activation = activations[0]
        gradient = activation.grad
        if gradient is None:
            raise RuntimeError("Grad-CAM gradient was not retained")
        weight = gradient.float().mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weight * activation.float()).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(CFG.image_size, CFG.image_size),
                            mode="bilinear", align_corners=False)
        cam = cam[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return pred, cam
    finally:
        handle.remove()


if CFG.run_gradcam:
    assert MODEL is not None and len(HEAD_MODELS) == 2
    # Coverage is the predeclared primary hybrid explanation; both are evaluated above.
    explanatory_head, explanatory_scaler = HEAD_MODELS["Sugeno_ANFIS_coverage"]
    MODEL.eval()
    explanatory_head.eval()
    examples = (VALID.groupby("dx", sort=False).head(1).set_index("dx")
                .reindex(CLASSES).reset_index())
    example_ds = LesionImages(examples, CFG.image_size, False, CFG.data_policy)
    fig, axes = plt.subplots(len(CLASSES), 4, figsize=(14, 3.1 * len(CLASSES)))
    headers = ("Image and predictions", "CNN Grad-CAM", "CNN–ANFIS Grad-CAM",
               "ANFIS heatmap")
    for col, header in enumerate(headers):
        axes[0, col].set_title(header, fontsize=10, pad=12, weight="bold")
    for row in range(len(example_ds)):
        tensor, actual, _ = example_ds[row]
        normalized = tensor.unsqueeze(0).to(DEVICE)
        cnn_pred, cnn_cam = gradcam_for_prediction(MODEL, normalized)
        hybrid_pred, hybrid_cam = gradcam_for_prediction(
            MODEL, normalized, explanatory_head, explanatory_scaler)
        image = np.clip(tensor.permute(1, 2, 0).numpy() * STD + MEAN, 0, 1)
        axes[row, 0].imshow(image)
        axes[row, 0].text(.03, .03,
            f"Actual: {CLASSES[actual]}\nCNN: {CLASSES[cnn_pred]}\nHybrid: {CLASSES[hybrid_pred]}",
            transform=axes[row, 0].transAxes, fontsize=8, va="bottom",
            bbox={"facecolor": "white", "alpha": .87, "edgecolor": "none"})
        axes[row, 1].imshow(image)
        axes[row, 1].imshow(cnn_cam, alpha=.42, cmap="magma")
        axes[row, 2].imshow(image)
        axes[row, 2].imshow(hybrid_cam, alpha=.42, cmap="magma")
        axes[row, 3].imshow(hybrid_cam, cmap="magma")
        for ax in axes[row]:
            ax.axis("off")
    fig.suptitle("CNN versus CNN–Sugeno ANFIS Grad-CAM (qualitative only)",
                 fontsize=13, y=.992)
    fig.subplots_adjust(left=.02, right=.98, bottom=.015, top=.95,
                        hspace=.35, wspace=.13)
    fig.savefig(OUT / "figures" / "cnn_vs_neurofuzzy_gradcam.png",
                dpi=155, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("Grad-CAM disabled or report-only mode")


## 12. Learning curves, software provenance and final exports

Export training/inference history, model configuration, dependency versions, immutable dataset manifest hash, model parameters, CPU/GPU environment, outputs and validated ZIP archives. Never report a result that this run did not compute; do not claim clinical validity or unseen test-set performance.


In [ ]:
def plot_training_history(path, output_name):
    if not path.exists():
        return
    frame = pd.read_csv(path)
    if "stage" in frame:
        groups = frame.groupby("stage", sort=False)
    else:
        groups = [("training", frame)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    for label, group in groups:
        axes[0].plot(group.epoch, group.train_loss, label=f"{label} training loss")
        if "internal_macro_f1" in group.columns and group.internal_macro_f1.notna().any():
            axes[1].plot(group.epoch, group.internal_macro_f1, label=f"{label} macro-F1")
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Training loss")
    axes[1].set(xlabel="Epoch", ylabel="Macro-F1", title="Training-only internal selection")
    for ax in axes:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / output_name, dpi=160)
    plt.show()
    plt.close(fig)


plot_training_history(HISTORY_PATH, "cnn_training_curves.png")
for head_name, _ in HEAD_SPECS:
    plot_training_history(OUT / "tables" / f"{head_name}_training_history.csv",
                          f"{head_name}_learning_curves.png")

provenance = {"course": "ACIT4620 Computational Intelligence", "group": "Group 1",
    "architecture": "ImageNet EfficientNet-B0 with 16D projection and 16-rule first-order Sugeno ANFIS",
    "generated_split_file": str(MANIFEST_PATH), "manifest_sha256": MANIFEST_SHA256,
    "raw_metadata_sha256": METADATA_SHA256,
    "metadata_file": str(META_PATH), "training_images": len(TRAIN),
    "validation_images": len(VALID), "sealed_test_images": len(TEST_IDS),
    "test_pixels_evaluated": bool(CFG.final_test_once), "run_configuration": asdict(CFG),
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "device": str(DEVICE),
    "new_split_generated_from_raw_metadata": True,
    "external_split_manifest_loaded": False,
    "train_grouped_cv_folds": CFG.cv_folds if CFG.run_cv else 0,
    "methodological_limitations": [
        "Not externally clinically validated",
        "Performance estimates are specific to this generated split and training protocol",
        "CNN and neuro-fuzzy comparative outputs must retain distinct model identities",
        "ANFIS epoch selection uses training-only internal data",
        ("The locked test was evaluated exactly once for the predeclared primary model"
         if CFG.final_test_once else
         "Test partition remains locked until the separately enabled final evaluation"),
    ]}
atomic_json(provenance, OUT / "reproducibility_provenance.json")

if RESULTS_PATH.exists():
    final_results = pd.read_csv(RESULTS_PATH)
    display(final_results[[c for c in ("split", "configuration", "accuracy", "macro_f1",
        "balanced_accuracy", "melanoma_recall", "macro_auroc_ovr",
        "macro_average_precision_ovr", "ece_15bins") if c in final_results]])
    assert not final_results.duplicated(subset=["configuration", "split"]).any()
    print("Completed model configurations:", final_results[["split", "configuration"]].values.tolist())
else:
    print("No new training/evaluation metrics generated in this run")

report_archive, checkpoint_archive = backup("FINAL")
print("Download BOTH final ZIP files from Kaggle outputs:")
print(report_archive)
print(checkpoint_archive)
print("Save the completed notebook version separately for code provenance")
if CFG.final_test_once:
    test_mark = OUT / "tables" / "sealed_test_evaluated.json"
    assert test_mark.exists(), "Required final test summary was not generated"
    assert (OUT / "figures" / f"{CFG.final_test_model}_test_confusion_matrices.png").exists()
    print("One-time sealed test verified:", json.loads(test_mark.read_text())["model"])



## 13. Submission and scientific reporting checks

- The generated split is an **output**, with image IDs, lesion-group assignments, fixed split seed and metadata fingerprint preserved for reproducibility.
- **Report only scores generated by this new study.** Only include metrics written by the executed notebook for this new split.
- The CNN head and each Sugeno model are independently identified in tables and figures; the pure CNN score is never presented as a hybrid score.
- The 16-rule fuzzy system uses learned Gaussian membership functions, normalized rule firing and first-order consequents. Fuzzy rules are not clinically validated concepts.
- All fitting, class imbalance handling, feature scaling, rule initialization and CNN checkpoint selection occur using **training data only**. The outer validation is a descriptive development evaluation. The sealed test remains unavailable until separately authorized.
- Grouped CV retrains CNN and ANFIS separately within each fold; mean ± sample SD across folds is not the same as across random initialization seeds.
- ImageNet initialization is transfer learning, not training the backbone from random weights. The notebook does **not** need weights or features from any previous HAM10000 experiment.
- If this research is submitted with an IEEE-style report, disclose methods and required tool assistance accurately according to institutional rules; code comments should explain functionality and design decisions, not authorship.

**Run All policy:** Training and grouped CV run from raw HAM10000; the predeclared coverage-regularized ANFIS is evaluated once on the test partition; the final test CSV, ROC/PR, reliability, blue confusion matrix, CV comparison plots, and 16-rule utilization heatmaps are packaged automatically. Do not use final-test results to retune architecture or hyperparameters.
